Aquí un ejemplo de DQN entrenando en el juego CartPole-v1 de OpenAI Gym:

In [ ]:
!pip install numpy --upgrade
!pip install gym --upgrade #Reinstall gym after updating numpy to ensure it uses the new version
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

In [ ]:
# Crear entorno
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

In [ ]:
# Definir la red neuronal DQN
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [ ]:
# Inicializar redes y optimizador
q_network = DQN(state_dim, action_dim)
target_network = DQN(state_dim, action_dim)
target_network.load_state_dict(q_network.state_dict())  # Copiar pesos

<All keys matched successfully>

In [ ]:
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [ ]:
# Parámetros de entrenamiento
gamma = 0.99
epsilon = 1.0  # Exploración inicial
epsilon_decay = 0.995
epsilon_min = 0.01
batch_size = 32
replay_buffer = deque(maxlen=10000)

In [ ]:
# Función para elegir acción (política epsilon-greedy)
def select_action(state):
    if np.random.rand() < epsilon:
        return env.action_space.sample()  # Explorar
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        q_values = q_network(state_tensor)
    return torch.argmax(q_values).item()  # Explotar

In [ ]:
# Entrenamiento de DQN
def train_dqn(episodes=500):
    global epsilon
    for episode in range(episodes):
        state = env.reset()[0]
        total_reward = 0
        done = False

        while not done:
            action = select_action(state)
            next_state, reward, done, _, _ = env.step(action)
            replay_buffer.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward

            # Entrenar la red si hay suficientes muestras
            if len(replay_buffer) > batch_size:
                batch = random.sample(replay_buffer, batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)

                states = torch.tensor(states, dtype=torch.float32)
                actions = torch.tensor(actions, dtype=torch.int64).unsqueeze(1)
                rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1)
                next_states = torch.tensor(next_states, dtype=torch.float32)
                dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1)

                # Calcular Q-target con la Red Objetivo
                q_values = q_network(states).gather(1, actions)
                with torch.no_grad():
                    max_next_q_values = target_network(next_states).max(dim=1, keepdim=True)[0]
                    q_targets = rewards + gamma * max_next_q_values * (1 - dones)

                loss = loss_fn(q_values, q_targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # Actualizar epsilon
        epsilon = max(epsilon * epsilon_decay, epsilon_min)

        # Copiar pesos a la Red Objetivo cada 10 episodios
        if episode % 10 == 0:
            target_network.load_state_dict(q_network.state_dict())

        print(f"Episode {episode}, Total Reward: {total_reward}")

In [ ]:
# Ejecutar entrenamiento
train_dqn()

AttributeError: module 'numpy' has no attribute 'bool8'